# Solution E1 · PCA redone on the test set: accuracy below chance

**Type**: Find the bug

## Objective

Find out why a classifier trained on the components of the training set gets 0.13 accuracy (worse than chance, with 3 classes) on the test set: the colleague **redid the PCA on the test set**, so the components of the test set are different axes (and with different signs) from those of the training set.
Fix it with `transform` and with a `Pipeline`.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_wine(return_X_y=True)
X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
scaler = StandardScaler().fit(X_tr_raw)
Z_tr, Z_te = scaler.transform(X_tr_raw), scaler.transform(X_te_raw)

## Commented solution

In [ ]:
train_pca = PCA(5).fit(Z_tr)
P_tr = train_pca.transform(Z_tr)
test_pca = PCA(5).fit(Z_te)                     # "to get the components of the test set"
P_te = test_pca.transform(Z_te)
model = LogisticRegression(max_iter=1000).fit(P_tr, y_tr)
bug_acc = model.score(P_te, y_te)
print(f"test accuracy: {bug_acc:.3f}")

**Where the bug is.** `PCA(5).fit(Z_te)` computes the components of the **test data**: they are other directions, and with signs that may be flipped. The logistic learned "a high coordinate 2 means class 0" in the frame of reference of the training set, but in the test set the same coordinate
can measure another direction (or the same one reversed): the predictions are systematically wrong, even worse than chance.

In [ ]:
cosines = [float(train_pca.components_[i] @ test_pca.components_[i]) for i in range(5)]
flipped_signs = int(sum(c < 0 for c in cosines))
print("dot products between the training and test components:", np.round(cosines, 2), "| components with the opposite direction:", flipped_signs)

In [ ]:
ok_acc = model.score(train_pca.transform(Z_te), y_te)                                   # the TRAINING components applied to the test set
pipeline = make_pipeline(StandardScaler(), PCA(5), LogisticRegression(max_iter=1000)).fit(X_tr_raw, y_tr)
pipeline_acc = pipeline.score(X_te_raw, y_te)
print(f"accuracy with transform: {ok_acc:.3f} | with the Pipeline: {pipeline_acc:.3f}")

## Check

In [ ]:
with verify():
    check(bug_acc < 0.5, "the colleague gets less than 0.5 with three classes", "don't modify the colleague's code")
    check(cosines is not None and len(cosines) == 5 and flipped_signs >= 2 and cosines[0] > 0.9 and min(cosines) < 0, "the first component is almost the same; at least two components have the opposite direction", "dot product between the rows of components_")
    check(ok_acc is not None and ok_acc > 0.9, "with transform (the same components as the training set) the accuracy goes back above 0.9", "train_pca.transform(Z_te)")
    check(pipeline_acc is not None and close(pipeline_acc, ok_acc), "the Pipeline gives exactly the same result", "make_pipeline(StandardScaler(), PCA(5), LogisticRegression())")
done()

## What you learned

- **A transformer is trained on the training set and applied (`transform`) to the test set.** Redoing `fit` or `fit_transform` on the test set creates a new frame of reference: for PCA it means different axes and arbitrary **signs** (the second and third components have dot product −0.87 and −0.84 with the training components),
  hence incompatible coordinates. Accuracy 0.13: worse than chance (0.4) because it is systematically wrong, not random.
- There is no error and no warning: the shapes are right, the values plausible. The warning sign is a result **worse than chance** from a model that worked on the training set.
- The rule holds for everything that learns from the data: `StandardScaler`, imputer, encoder, PCA. The `Pipeline` makes it impossible to violate, because `predict` calls only `transform`.
- A milder variant is to `fit` the PCA on **all** the data (training and test) before the split: the signs stay coherent and the effect is small, but the test set influenced the components (mild leakage). The pipeline avoids this too.

## Common pitfall

Thinking that "fitting the PCA on the test set" is fine because it doesn't use the labels. The components depend on the distribution of the data: applying a model trained in one frame of reference to points expressed in another makes no sense.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Why can't you do fit_transform on the test set too?</summary>

Because fit learns parameters from the data (means, deviations, components) and the downstream model was trained with the parameters of the training set. Redoing the fit on the test set would change the frame of reference (with PCA: different axes and signs), so the scores would not be comparable with the ones the model saw.
You use transform with the objects already trained, and a Pipeline guarantees it by construction.

</details>